In [ ]:
import os
import re
import glob
import pandas as pd
from pathlib import Path
from slap2_processing.src import generate_instrument_json as gij
from slap2_processing.src.generate_acquisition_json import Slap2VCOAcquisitionEtl

In [ ]:
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
summary_path = glob.glob(os.path.join(basepath,'**summary.xlsx'))[0]
summary_df = pd.read_excel(summary_path,sheet_name='subjects')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [ ]:
session_df

In [ ]:
def find_session_root_subdir(session_dir: Path, mouse: int | str) -> Path:
    """
    Find the subdirectory inside `session_dir` whose name matches:
        nnnnnn_yyyy-mm-dd_hh-mm-ss
    where nnnnnn == mouse id (6 digits).
    Returns the matching Path, or raises ValueError with a helpful message.
    """
    mouse = str(mouse)
    pattern = re.compile(rf"^{re.escape(mouse)}_\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}}$")

    matches = [p for p in session_dir.iterdir() if p.is_dir() and pattern.match(p.name)]
    if len(matches) == 0:
        raise ValueError(f"No matching destination subdir in {session_dir}")
    if len(matches) > 1:
        raise ValueError(f"Multiple matching destination subdirs in {session_dir}: {[m.name for m in matches]}")
    return matches[0]

### Generate instrument.json

In [ ]:
for idx, row in session_df.iterrows():
    try:
        sess = find_session_root_subdir(Path(row['session_dir']), mouse=row['subject_id'])
    except Exception:
        sess = Path(row['session_dir'])

    out_path = Path(sess) / "instrument.json"
    print(f"Attempting to generate instrument.json for {Path(sess).stem}")

    try:
        gij.generate_instrument_json(
            Path(sess),
            use_vimba=(row['camera_type'] == 'vimba')
        )

        if out_path.exists():
            print(f"SUCCESS: wrote {out_path}")
        else:
            print(f"NO FILE FOUND AFTER RUN: expected {out_path}")

    except BaseException as e:
        print(f"FAILED: {type(e).__name__}: {e}")

    print()

### Get stimulus parameters

In [ ]:
stim_path = r"C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\Aind.Behavior.ChangeDetection\src\stimuli\images_B"

In [ ]:
image_names = os.listdir(stim_path)
image_names

### Generate acquisition.json

In [ ]:
mouse = 803121
process_dirs = session_df[(session_df['subject_id']==mouse)
                          &(session_df['session_type']!='expression_check')
                          &(session_df['session_type']!='volume_imaging')]['session_dir'].values
session_dirs = [find_session_root_subdir(Path(sess),mouse) for sess in process_dirs]
results = []

for sess_dir in session_dirs:
    try:
        etl = Slap2VCOAcquisitionEtl(
            asset_dir=sess_dir,
            output_dir=sess_dir,      
            timing_source="harp",     
            timing_source_column=None,
            save_images=False,
        )
        etl.run_job()

        results.append({
            "session_dir": str(sess_dir),
            "status": "success",
            "error": ""
        })

    except Exception as e:
        results.append({
            "session_dir": str(sess_dir),
            "status": "failed",
            "error": str(e)
        })

results_df = pd.DataFrame(results)
results_df